# Fase 5: ¿dónde mira la red y qué tanto aguanta?

Las fases anteriores dejaron una pregunta abierta. Sabemos que la red acierta el 97%, y sabemos
que **no lo hace apoyándose en la geometría fina del núcleo**, porque esa información ni la tiene
codificada ni le sirvió cuando se la dimos. Entonces, ¿en qué se está fijando?

Esta fase responde dos cosas, y las dos con números en vez de impresiones:

1. **Dónde mira.** Se generan mapas de calor que muestran qué zona de la foto pesó en la decisión,
   y como tenemos el contorno real del núcleo, podemos medir qué porcentaje de esa atención cae
   dentro de él. Lo habitual en la literatura es mostrar cuatro mapas bonitos y decir "se ve que
   mira el núcleo". Nosotros lo medimos en las 4049 células.

2. **Qué tanto aguanta.** Se le meten las fotos con los colores cambiados, desenfocadas y
   comprimidas, simulando otro laboratorio con otra tinción y otro microscopio. Un modelo que se
   cae con un cambio de tono no sirve fuera del hospital donde se tomaron las fotos.

Y al final se cruzan las dos: cuando el modelo se equivoca por las perturbaciones, ¿es porque
dejó de mirar donde debía?

## Preparativos

In [ ]:
import torch
print("GPU disponible:", torch.cuda.is_available())
print("Modelo:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "sin GPU")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
from google.colab import userdata
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')

In [ ]:
import pandas as pd, numpy as np, kagglehub
from sklearn.model_selection import StratifiedGroupKFold

DEST = "/content/drive/MyDrive/sipakmed"
PESOS = f"{DEST}/pesos_fase5"
os.makedirs(PESOS, exist_ok=True)

df = pd.read_csv(f"{DEST}/indice_celulas.csv").reset_index(drop=True)

path = kagglehub.dataset_download("prahladmehandiratta/cervical-cancer-largest-dataset-sipakmed")
base_viejo = df["img"].iloc[0].split("/im_")[0]
for col in ["img", "cyt", "nuc"]:
    df[col] = df[col].str.replace(base_viejo, path, regex=False)

SEMILLA = 42
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEMILLA)
df["fold_lamina"] = -1
for k, (_, val) in enumerate(sgkf.split(df, df["clase"], groups=df["lamina"])):
    df.loc[df.index[val], "fold_lamina"] = k

CLASES = sorted(df["clase"].unique())
IDX = {c: i for i, c in enumerate(CLASES)}
Y = df["clase"].map(IDX).values
print("Células:", len(df), "| imágenes encontradas:", df["img"].apply(os.path.exists).all())

In [ ]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import v2

print("Cargando imágenes...")
CACHE = np.zeros((len(df), 224, 224, 3), dtype=np.uint8)
for i, ruta in enumerate(df["img"]):
    CACHE[i] = np.array(Image.open(ruta).convert("RGB").resize((224, 224), Image.BILINEAR))
    if i % 500 == 0:
        print(f"  {i}/{len(df)}", end="\r")
print(f"Listo: {CACHE.nbytes/1e6:.0f} MB")

MEDIA = [0.485, 0.456, 0.406]
DESV  = [0.229, 0.224, 0.225]

tf_train = v2.Compose([
    v2.RandomHorizontalFlip(), v2.RandomVerticalFlip(), v2.RandomRotation(20),
    v2.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15, hue=0.03),
    v2.ToDtype(torch.float32, scale=True), v2.Normalize(MEDIA, DESV),
])
tf_eval = v2.Compose([v2.ToDtype(torch.float32, scale=True), v2.Normalize(MEDIA, DESV)])


class Celulas(Dataset):
    def __init__(self, posiciones, tf):
        self.pos = np.asarray(posiciones)
        self.tf = tf

    def __len__(self):
        return len(self.pos)

    def __getitem__(self, i):
        p = self.pos[i]
        return self.tf(torch.from_numpy(CACHE[p]).permute(2, 0, 1)), Y[p], p

## Las máscaras del núcleo

Los contornos vienen en coordenadas del recorte original, y las fotos las escalamos a 224x224.
Hay que escalar los contornos igual, si no las máscaras quedarían corridas y toda la medición
saldría mal.

In [ ]:
import cv2

def mascaras(p):
    """Devuelve mascara del nucleo y de la celula completa, a escala 224x224."""
    w, h = df.loc[p, "w"], df.loc[p, "h"]
    sx, sy = 224.0 / w, 224.0 / h

    def poly(ruta):
        pts = np.loadtxt(ruta, delimiter=",").reshape(-1, 2)
        pts[:, 0] *= sx; pts[:, 1] *= sy       # misma escala que la foto
        return pts.astype(np.int32)

    m_nuc = np.zeros((224, 224), np.uint8)
    m_cyt = np.zeros((224, 224), np.uint8)
    cv2.fillPoly(m_nuc, [poly(df.loc[p, "nuc"])], 1)
    cv2.fillPoly(m_cyt, [poly(df.loc[p, "cyt"])], 1)
    return m_nuc, m_cyt


# Comprobacion rapida: el nucleo debe ocupar una fraccion chica y estar dentro de la celula
n, c = mascaras(0)
print(f"Núcleo: {n.mean():.3f} de la imagen | célula: {c.mean():.3f}")
print("Núcleo contenido en la célula:", bool(((n == 1) & (c == 0)).sum() < n.sum() * 0.05))

## Entrenar y guardar los modelos

Cinco pliegues, la misma red de siempre. Esta vez guardamos los pesos en Drive, porque los vamos a
necesitar dos veces más: para los mapas de calor y para las pruebas de aguante. Así no hay que
reentrenar.

In [ ]:
import torch.nn as nn, random, time
from torchvision import models
from sklearn.metrics import f1_score

EPOCAS = 12
LOTE = 64
LR = 3e-4
DEV = "cuda" if torch.cuda.is_available() else "cpu"


def fijar_semilla(s):
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def crear_red():
    base = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
    base.classifier = nn.Identity()
    cabeza = nn.Sequential(nn.Linear(1280, 256), nn.ReLU(), nn.Dropout(0.3),
                           nn.Linear(256, len(CLASES)))
    return base.to(DEV), cabeza.to(DEV)


pred_base = np.full(len(df), -1)

for k in range(5):
    archivo = f"{PESOS}/fold{k}.pt"
    if os.path.exists(archivo):
        print(f"  pliegue {k} ya entrenado, se reutiliza")
        continue

    t0 = time.time()
    fijar_semilla(SEMILLA)
    tr = np.where(df["fold_lamina"].values != k)[0]

    dl_tr = DataLoader(Celulas(tr, tf_train), batch_size=LOTE, shuffle=True,
                       num_workers=2, pin_memory=True, drop_last=True)

    base, cabeza = crear_red()
    opt = torch.optim.AdamW(list(base.parameters()) + list(cabeza.parameters()),
                            lr=LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCAS)
    criterio = nn.CrossEntropyLoss()
    scaler = torch.amp.GradScaler("cuda")

    for ep in range(EPOCAS):
        base.train(); cabeza.train()
        for x, y, _ in dl_tr:
            x, y = x.to(DEV), y.to(DEV)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda"):
                loss = criterio(cabeza(base(x)), y)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
        sched.step()

    torch.save({"base": base.state_dict(), "cabeza": cabeza.state_dict()}, archivo)
    del base, cabeza; torch.cuda.empty_cache()
    print(f"  pliegue {k} entrenado y guardado ({time.time()-t0:.0f}s)")


def cargar(k):
    base, cabeza = crear_red()
    est = torch.load(f"{PESOS}/fold{k}.pt", map_location=DEV)
    base.load_state_dict(est["base"]); cabeza.load_state_dict(est["cabeza"])
    base.eval(); cabeza.eval()
    return base, cabeza

## Los mapas de calor

Un mapa de calor señala qué zonas de la foto empujaron la decisión. La técnica mira la última capa
de la red, donde cada punto todavía guarda su posición en la imagen, y pondera cada canal según
cuánto habría cambiado la respuesta si ese canal cambiara.

Se calcula para cada célula usando **el modelo que no la vio durante el entrenamiento**, igual que
todo lo demás en este trabajo.

In [ ]:
import torch.nn.functional as Fn

def mapas_calor(base, cabeza, x):
    """Devuelve un mapa de 224x224 por imagen del lote, y la clase predicha."""
    guardado = {}

    def hook(m, entrada, salida):
        guardado["act"] = salida

    h = base.features[-1].register_forward_hook(hook)
    z = base(x)
    s = cabeza(z)
    h.remove()

    # Gradiente de la clase predicha respecto a la ultima capa con posicion
    y_pred = s.argmax(1).detach()
    objetivo = s.gather(1, y_pred.view(-1, 1)).sum()
    grad = torch.autograd.grad(objetivo, guardado["act"])[0]

    peso = grad.mean(dim=(2, 3), keepdim=True)
    cam = torch.relu((peso * guardado["act"]).sum(dim=1, keepdim=True))
    cam = Fn.interpolate(cam, size=(224, 224), mode="bilinear", align_corners=False)
    return cam[:, 0].detach().cpu().numpy(), s.argmax(1).detach().cpu().numpy()

## La medida: ¿cuánta atención cae en el núcleo?

Aquí está el aporte de esta fase. Si simplemente midiéramos qué porcentaje del mapa de calor cae
dentro del núcleo, un modelo que se activa por toda la imagen saldría bien parado solo porque el
núcleo es parte de esa imagen. Hay que descontar el tamaño.

Por eso dividimos entre la fracción de imagen que ocupa el núcleo:

- **Valor 1:** el modelo reparte la atención por igual, no privilegia el núcleo.
- **Mayor que 1:** concentra en el núcleo más de lo que le tocaría por tamaño.
- **Menor que 1:** mira sobre todo fuera del núcleo.

Lo calculamos también para el citoplasma y para el fondo, así se ve el reparto completo.

In [ ]:
ATEN_NUC = np.full(len(df), np.nan)
ATEN_CIT = np.full(len(df), np.nan)
ATEN_FON = np.full(len(df), np.nan)

for k in range(5):
    t0 = time.time()
    base, cabeza = cargar(k)
    te = np.where(df["fold_lamina"].values == k)[0]
    dl = DataLoader(Celulas(te, tf_eval), batch_size=32, shuffle=False, num_workers=2)

    for x, y, pos in dl:
        cams, preds = mapas_calor(base, cabeza, x.to(DEV))
        pred_base[pos.numpy()] = preds

        for j, p in enumerate(pos.numpy()):
            cam = cams[j]
            total = cam.sum()
            if total <= 0:
                continue
            m_nuc, m_cyt = mascaras(p)
            m_cit = np.clip(m_cyt.astype(int) - m_nuc.astype(int), 0, 1).astype(np.uint8)
            m_fon = 1 - m_cyt

            for mascara, destino in [(m_nuc, ATEN_NUC), (m_cit, ATEN_CIT), (m_fon, ATEN_FON)]:
                area = mascara.mean()
                if area > 0:
                    destino[p] = (cam[mascara == 1].sum() / total) / area

    del base, cabeza; torch.cuda.empty_cache()
    print(f"  pliegue {k} listo ({time.time()-t0:.0f}s)")

print("\nmacro F1:", round(f1_score(Y, pred_base, average='macro'), 4))
print(f"Atención en el núcleo:    {np.nanmean(ATEN_NUC):.2f}")
print(f"Atención en el citoplasma: {np.nanmean(ATEN_CIT):.2f}")
print(f"Atención en el fondo:      {np.nanmean(ATEN_FON):.2f}")

## ¿Los errores miran en otro lado?

La hipótesis interesante: cuando el modelo se equivoca, quizá es porque se fijó en el lugar
equivocado. Si la atención sobre el núcleo es más baja en los errores que en los aciertos, tenemos
una relación entre *dónde mira* y *si acierta*, que es bastante más de lo que suele mostrarse.

Usamos una prueba que no asume que los datos sigan una campana, porque estos valores no la siguen.

In [ ]:
from scipy.stats import mannwhitneyu
import matplotlib.pyplot as plt

acierta = (pred_base == Y)
val = ~np.isnan(ATEN_NUC)

a = ATEN_NUC[val & acierta]
b = ATEN_NUC[val & ~acierta]
u, p = mannwhitneyu(a, b)

print(f"Aciertos  (n={len(a)}): atención al núcleo {np.median(a):.2f} (mediana)")
print(f"Errores   (n={len(b)}): atención al núcleo {np.median(b):.2f} (mediana)")
print(f"p = {p:.2e} ->", "hay diferencia" if p < 0.05 else "no hay diferencia clara")

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].boxplot([a, b], labels=["aciertos", "errores"], showfliers=False)
axes[0].axhline(1, color="r", ls="--", lw=1)
axes[0].set_ylabel("Atención al núcleo")
axes[0].set_title("¿Mira distinto cuando falla?")

datos = [ATEN_NUC[val & (df["clase"].values == c)] for c in CLASES]
axes[1].boxplot(datos, labels=[c[:12] for c in CLASES], showfliers=False)
axes[1].axhline(1, color="r", ls="--", lw=1)
axes[1].tick_params(axis="x", rotation=45, labelsize=8)
axes[1].set_title("Atención al núcleo por clase")
plt.tight_layout(); plt.show()

## Verlo con los ojos

Unos cuantos ejemplos, con el contorno real del núcleo dibujado encima del mapa de calor. Sirve
para confirmar que la medida está midiendo lo que creemos.

In [ ]:
ejemplos = []
for c in CLASES:
    cand = np.where((df["clase"].values == c) & val)[0]
    ejemplos.append(cand[np.argmax(ATEN_NUC[cand])])   # el que mas mira el nucleo
    ejemplos.append(cand[np.argmin(ATEN_NUC[cand])])   # el que menos

base, cabeza = cargar(0)
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
for j, p in enumerate(ejemplos):
    x = tf_eval(torch.from_numpy(CACHE[p]).permute(2, 0, 1)).unsqueeze(0).to(DEV)
    cam, _ = mapas_calor(base, cabeza, x)
    cam = cam[0] / (cam[0].max() + 1e-8)

    m_nuc, _ = mascaras(p)
    cont, _ = cv2.findContours(m_nuc, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    vis = CACHE[p].copy()
    cv2.drawContours(vis, cont, -1, (255, 255, 0), 2)

    ax = axes[j % 2, j // 2]
    ax.imshow(vis); ax.imshow(cam, cmap="jet", alpha=0.45)
    ax.set_title(f"{df.loc[p,'clase'][:11]}\naten={ATEN_NUC[p]:.2f}", fontsize=8)
    ax.axis("off")
plt.tight_layout(); plt.show()
del base, cabeza; torch.cuda.empty_cache()

## Segunda parte: ¿aguanta otro laboratorio?

Las fotos de este dataset salieron todas del mismo microscopio y con la misma tinción. En otro
hospital el tono es distinto, el enfoque no es idéntico y las imágenes suelen guardarse
comprimidas.

Simulamos esas tres cosas **solo al evaluar**, sin reentrenar nada. Los modelos son los mismos que
ya están guardados.

In [ ]:
def perturbar(img, tipo, nivel):
    """img es uint8 de 224x224x3. Devuelve la version alterada."""
    if tipo == "tinción":
        hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV).astype(np.int16)
        hsv[..., 0] = (hsv[..., 0] + nivel) % 180            # corre el tono
        hsv[..., 1] = np.clip(hsv[..., 1] * (1 + nivel / 60), 0, 255)
        return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2RGB)

    if tipo == "desenfoque":
        return cv2.GaussianBlur(img, (0, 0), sigmaX=nivel)

    if tipo == "compresión":
        ok, buf = cv2.imencode(".jpg", img[:, :, ::-1],
                               [int(cv2.IMWRITE_JPEG_QUALITY), nivel])
        return cv2.imdecode(buf, cv2.IMREAD_COLOR)[:, :, ::-1]

    return img


class CelulasPerturbadas(Dataset):
    def __init__(self, posiciones, tipo, nivel):
        self.pos = np.asarray(posiciones)
        self.tipo, self.nivel = tipo, nivel

    def __len__(self):
        return len(self.pos)

    def __getitem__(self, i):
        p = self.pos[i]
        img = perturbar(CACHE[p], self.tipo, self.nivel)
        return tf_eval(torch.from_numpy(np.ascontiguousarray(img)).permute(2, 0, 1)), Y[p], p


# Vista rapida de como se ven las perturbaciones
fig, axes = plt.subplots(1, 5, figsize=(15, 3.2))
for ax, (t, n) in zip(axes, [(None, 0), ("tinción", 10), ("tinción", 25),
                             ("desenfoque", 2.5), ("compresión", 15)]):
    ax.imshow(CACHE[0] if t is None else perturbar(CACHE[0], t, n))
    ax.set_title("original" if t is None else f"{t} {n}", fontsize=9)
    ax.axis("off")
plt.tight_layout(); plt.show()

In [ ]:
CONDICIONES = [("original", None, 0),
               ("tinción", "tinción", 8), ("tinción", "tinción", 16), ("tinción", "tinción", 25),
               ("desenfoque", "desenfoque", 1.0), ("desenfoque", "desenfoque", 2.0),
               ("desenfoque", "desenfoque", 3.0),
               ("compresión", "compresión", 50), ("compresión", "compresión", 25),
               ("compresión", "compresión", 10)]

filas = []
for etiqueta, tipo, nivel in CONDICIONES:
    pred = np.full(len(df), -1)
    for k in range(5):
        base, cabeza = cargar(k)
        te = np.where(df["fold_lamina"].values == k)[0]
        dl = DataLoader(CelulasPerturbadas(te, tipo, nivel), batch_size=128,
                        shuffle=False, num_workers=2)
        with torch.no_grad():
            for x, y, pos in dl:
                with torch.amp.autocast("cuda"):
                    s = cabeza(base(x.to(DEV)))
                pred[pos.numpy()] = s.argmax(1).cpu().numpy()
        del base, cabeza; torch.cuda.empty_cache()

    f1 = f1_score(Y, pred, average="macro")
    filas.append({"perturbación": etiqueta, "nivel": nivel, "macro_f1": f1})
    print(f"  {etiqueta:12s} nivel {nivel:5} -> F1 {f1:.4f}")

rob = pd.DataFrame(filas)
base_f1 = rob.loc[0, "macro_f1"]
rob["caída_%"] = ((base_f1 - rob["macro_f1"]) / base_f1 * 100).round(2)
rob

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
for tipo in ["tinción", "desenfoque", "compresión"]:
    sub = rob[rob["perturbación"] == tipo]
    ax.plot(range(1, len(sub) + 1), sub["macro_f1"], "o-", label=tipo)
ax.axhline(base_f1, color="k", ls="--", lw=1, label="sin perturbar")
ax.set_xticks([1, 2, 3]); ax.set_xticklabels(["suave", "medio", "fuerte"])
ax.set_ylabel("macro F1"); ax.set_xlabel("intensidad de la perturbación")
ax.legend(); ax.set_title("Qué tanto aguanta el modelo")
plt.tight_layout(); plt.show()

## Cruzar las dos partes

La pregunta que une todo: cuando el modelo empeora con las perturbaciones, ¿es porque dejó de
mirar el núcleo?

Repetimos la medición de atención con la perturbación de tinción más fuerte y comparamos contra la
original. Si la atención se desploma junto con el acierto, tenemos una explicación de la fragilidad
en vez de solo constatarla.

In [ ]:
ATEN_PERT = np.full(len(df), np.nan)

for k in range(5):
    base, cabeza = cargar(k)
    te = np.where(df["fold_lamina"].values == k)[0]
    dl = DataLoader(CelulasPerturbadas(te, "tinción", 25), batch_size=32,
                    shuffle=False, num_workers=2)

    for x, y, pos in dl:
        cams, _ = mapas_calor(base, cabeza, x.to(DEV))

        for j, p in enumerate(pos.numpy()):
            cam = cams[j]
            if cam.sum() <= 0:
                continue
            m_nuc, _ = mascaras(p)
            area = m_nuc.mean()
            if area > 0:
                ATEN_PERT[p] = (cam[m_nuc == 1].sum() / cam.sum()) / area

    del base, cabeza; torch.cuda.empty_cache()
    print(f"  pliegue {k} listo")

ok = ~np.isnan(ATEN_NUC) & ~np.isnan(ATEN_PERT)
print(f"\nAtención al núcleo, original:      {ATEN_NUC[ok].mean():.3f}")
print(f"Atención al núcleo, tinción alterada: {ATEN_PERT[ok].mean():.3f}")
print(f"Cambio: {(ATEN_PERT[ok].mean() - ATEN_NUC[ok].mean()):+.3f}")

plt.figure(figsize=(5.5, 5.5))
plt.scatter(ATEN_NUC[ok], ATEN_PERT[ok], s=5, alpha=0.25)
lim = [0, np.percentile(ATEN_NUC[ok], 99)]
plt.plot(lim, lim, "r--", lw=1); plt.xlim(lim); plt.ylim(lim)
plt.xlabel("Atención al núcleo, original")
plt.ylabel("Atención al núcleo, con tinción alterada")
plt.tight_layout(); plt.show()

## Guardar

In [ ]:
rob.to_csv(f"{DEST}/robustez_fase5.csv", index=False)

sal = df[["stem", "clase", "lamina", "fold_lamina"]].copy()
sal["pred"] = pred_base
sal["aten_nucleo"] = ATEN_NUC
sal["aten_citoplasma"] = ATEN_CIT
sal["aten_fondo"] = ATEN_FON
sal["aten_nucleo_perturbado"] = ATEN_PERT
sal.to_csv(f"{DEST}/atencion_fase5.csv", index=False)
print("Guardado en", DEST)

## Resumen de la fase 5

Lo que aporta:

- **Convierte los mapas de calor en un número.** En vez de mostrar cuatro ejemplos elegidos a mano
  y afirmar que el modelo mira el núcleo, se mide en las 4049 células y se descuenta el tamaño del
  núcleo para que la medida no premie a los modelos que se activan en todas partes.
- **Relaciona dónde mira con si acierta**, comparando la atención en aciertos y en errores.
- **Mide qué tanto aguanta** cambios de tinción, desenfoque y compresión, que es la diferencia
  entre un modelo que funciona en el laboratorio de origen y uno que sirve en otro hospital.
- **Une las dos cosas:** si el acierto cae con la perturbación y la atención también, la
  fragilidad tiene una explicación y no es solo un número malo.

Junto con las fases anteriores, el trabajo queda con una historia completa: cuánto se infla el
resultado por el reparto, qué información usa realmente la red, y qué tan lejos llega fuera de sus
propias condiciones.